# Setup

In [1]:
"""
Chunking exploration.

Goal: split the 40 cleaned section files into overlapping ~800-token chunks
with full provenance metadata, ready for embedding in Week 2.
"""
from pathlib import Path
from collections import Counter
import statistics

import tiktoken
from finintel.retrieval.chunker import (
    chunk_section,
    chunk_processed_corpus,
    write_chunks_jsonl,
)

%load_ext autoreload
%autoreload 2

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
encoding = tiktoken.get_encoding("cl100k_base")

n_section_files = len(list(PROCESSED_DIR.rglob("*.txt")))
print(f"Processed dir: {PROCESSED_DIR.relative_to(PROJECT_ROOT)}")
print(f"Section files found: {n_section_files}")  # should be 40

Processed dir: data\processed
Section files found: 40


# Chunk one section to see the shape

In [2]:
sample_path = PROCESSED_DIR / "AAPL" / "10-K" / "0000320193-25-000079" / "risk_factors.txt"
text = sample_path.read_text(encoding="utf-8")
print(f"Source: {len(text):,} chars, {len(encoding.encode(text)):,} tokens\n")

chunks = chunk_section(
    text,
    ticker="AAPL", filing_type="10-K",
    accession="0000320193-25-000079", section="risk_factors",
)
print(f"Produced {len(chunks)} chunks\n")
for c in chunks[:5]:
    print(f"  {c.chunk_id}: {c.n_tokens:>4} tokens, {len(c.text):>5} chars")
print(f"  ...")
for c in chunks[-2:]:
    print(f"  {c.chunk_id}: {c.n_tokens:>4} tokens, {len(c.text):>5} chars")

Source: 68,044 chars, 11,602 tokens

Produced 18 chunks

  AAPL_10-K_0000320193-25-000079_risk_factors_000:  778 tokens,  4520 chars
  AAPL_10-K_0000320193-25-000079_risk_factors_001:  775 tokens,  4417 chars
  AAPL_10-K_0000320193-25-000079_risk_factors_002:  793 tokens,  4746 chars
  AAPL_10-K_0000320193-25-000079_risk_factors_003:  712 tokens,  4298 chars
  AAPL_10-K_0000320193-25-000079_risk_factors_004:  784 tokens,  4806 chars
  ...
  AAPL_10-K_0000320193-25-000079_risk_factors_016:  793 tokens,  4456 chars
  AAPL_10-K_0000320193-25-000079_risk_factors_017:  633 tokens,  3517 chars


# Inspect a single chunk

In [3]:
print(chunks[5].text)
print(f"\n--- {chunks[5].chunk_id} | {chunks[5].n_tokens} tokens ---")

. Future operating results depend upon the Company’s ability to obtain components in sufficient quantities on commercially reasonable terms. Because the Company currently obtains certain components from single or limited sources, the Company is subject to significant supply and pricing risks. Many components, including those that are available from multiple sources, are at times subject to industry-wide shortages and significant commodity pricing fluctuations that can materially adversely affect the Company’s business, results of operations, financial condition and stock price. For example, the global semiconductor industry has in the past experienced high demand and shortages of supply, which adversely affected the Company’s ability to obtain sufficient quantities of components and products on commercially reasonable terms, or at all. Such disruptions could occur in the future. Additionally, the Company’s new products often utilize custom components available from only one source. Whe

# Token distribution (sanity check)

In [4]:
token_counts = [c.n_tokens for c in chunks]
print(f"Total chunks:    {len(chunks)}")
print(f"Mean tokens:     {statistics.mean(token_counts):.0f}")
print(f"Median tokens:   {statistics.median(token_counts):.0f}")
print(f"Min / Max:       {min(token_counts)} / {max(token_counts)}")
print(f"Std deviation:   {statistics.stdev(token_counts):.0f}")

Total chunks:    18
Mean tokens:     770
Median tokens:   788
Min / Max:       633 / 800
Std deviation:   42


# Chunk the entire corpus

In [5]:
all_chunks = list(chunk_processed_corpus(PROCESSED_DIR))
print(f"Total chunks across corpus: {len(all_chunks)}\n")

print(f"{'Ticker':<8} {'Section':<14} {'Chunks':>7}")
print("-" * 32)
by_ticker_section = Counter((c.ticker, c.section) for c in all_chunks)
for (ticker, section), count in sorted(by_ticker_section.items()):
    print(f"{ticker:<8} {section:<14} {count:>7}")

Total chunks across corpus: 1259

Ticker   Section         Chunks
--------------------------------
AAPL     mda                 23
AAPL     risk_factors        73
GOOGL    mda                 70
GOOGL    risk_factors        88
JPM      mda                572
JPM      risk_factors       139
MSFT     mda                 66
MSFT     risk_factors        76
TSLA     mda                 65
TSLA     risk_factors        87


# Save to disk

In [6]:
output_path = PROJECT_ROOT / "data" / "chunks.jsonl"
n = write_chunks_jsonl(all_chunks, output_path)

size_mb = output_path.stat().st_size / (1024 * 1024)
print(f"Wrote {n:,} chunks to {output_path.relative_to(PROJECT_ROOT)}")
print(f"File size: {size_mb:.2f} MB")

Wrote 1,259 chunks to data\chunks.jsonl
File size: 4.85 MB
